In [ ]:
import torch
import torch.nn as nn

In [ ]:
class AUDeepfakeDetector(nn.Module):
    def __init__(self, input_dim=17, seq_len=64, hidden_dim=64, num_classes=1):
        super(AUDeepfakeDetector, self).__init__()

        # 1. 1D CNN: 지역적인 AU 패턴 변화(미세한 떨림 등) 추출
        # 입력 형태: (Batch, Channels, Length)를 맞추기 위해 forward에서 transpose 필요
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2) # Length: 64 -> 32
        
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        # 2nd pool 적용 시 Length: 32 -> 16

        # 2. Bi-LSTM: 표정 변화의 시간적 비일관성 탐지
        self.lstm = nn.LSTM(
            input_size=64, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True,
            dropout=0.3
        )

        # 3. Classifier
        # Bi-LSTM이므로 출력 차원은 hidden_dim * 2
        self.fc1 = nn.Linear(hidden_dim * 2, 32)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(32, num_classes)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape from dataset: (B, 64, 17)
        # Conv1d는 (B, C, L) 포맷을 요구하므로 transpose
        x = x.transpose(1, 2)  # -> (B, 17, 64)

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)       # -> (B, 32, 32)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)       # -> (B, 64, 16)

        # LSTM 입력을 위해 (B, L, C) 포맷으로 복구
        x = x.transpose(1, 2)  # -> (B, 16, 64)

        # lstm_out shape: (B, L, hidden_dim * 2)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # 시퀀스의 마지막 타임스텝 출력값만 분류기에 전달
        last_hidden = lstm_out[:, -1, :] # -> (B, hidden_dim * 2)

        out = self.fc1(last_hidden)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        # 학습 시 BCEWithLogitsLoss를 사용한다면 sigmoid를 제거하는 것이 수치적으로 안정적임.
        # 추론 시 확률값이 필요하므로 일단 적용해 둠.
        return self.sigmoid(out)

In [ ]:
# 텐서 형태 확인용 더미 테스트
if __name__ == "__main__":
    batch_size = 8
    # 전처리 완료된 (B, T, C) 형태의 AU 텐서
    mock_au_input = torch.rand(batch_size, 64, 17) 
    model = AUDeepfakeDetector()
    
    preds = model(mock_au_input)
    print("Predictions shape:", preds.shape) # 기대값: (8, 1)